# APOD RAG Pipeline 

This notebook builds the RAG pipeline step by step:
1. Load the model
2. Understand embeddings
3. Explore similarity
4. Store in Chroma
5. Search
6. Connect to Claude

The goal is to understand each piece before it moves to the pipeline script.

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "A spiral galaxy colliding with its neighbor",
    "Two galaxies merging together",
    "Recipe for chocolate cake",
]

vectors = model.encode(sentences)
print(vectors.shape)  # (3, 384) — 3 sentences, 384 numbers each

/Users/christinahuynh/anaconda3/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)


## 1. Load the model

`SentenceTransformer` is a library that wraps pre-trained neural networks
for converting text into vectors. `all-MiniLM-L6-v2` is a small, fast model
that produces 384-dimensional vectors. It downloads once (~80MB) and caches locally.

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded!")
print(f"Vector size: {model.get_sentence_embedding_dimension()} dimensions")

Model loaded!
Vector size: 384 dimensions


## 2. What does an embedding look like?

`model.encode()` takes text and returns a vector — a list of numbers
that represents the meaning of the text. The numbers themselves are not
human readable but similar texts produce similar vectors.

In [5]:
text = "A spiral galaxy colliding with its neighbor"
vector = model.encode(text)

print(f"Input text: {text}")
print(f"Vector shape: {vector.shape}")
print(f"First 10 numbers: {vector[:10].round(4)}")

Input text: A spiral galaxy colliding with its neighbor
Vector shape: (384,)
First 10 numbers: [-0.0614  0.0042  0.0076 -0.0289 -0.1134 -0.0204 -0.06    0.0179  0.0453
 -0.0488]


## 3. Similarity between vectors

Cosine similarity measures how similar two vectors are.
Returns a value between -1 and 1:
- close to 1 = very similar meaning
- close to 0 = unrelated
- close to -1 = opposite meaning

This is how Chroma finds relevant entries — it finds vectors
closest to your question vector.

In [6]:
from sentence_transformers import util

v1 = model.encode("black hole event horizon")
v2 = model.encode("collapsed star with strong gravity")
v3 = model.encode("recipe for chocolate cake")

sim_12 = util.cos_sim(v1, v2).item()
sim_13 = util.cos_sim(v1, v3).item()

print(f"'black hole' vs 'collapsed star': {sim_12:.4f}  ← similar meaning")
print(f"'black hole' vs 'chocolate cake': {sim_13:.4f}  ← very different")

'black hole' vs 'collapsed star': 0.2726  ← similar meaning
'black hole' vs 'chocolate cake': 0.1430  ← very different


## 4. Load APOD gold data

Load the enriched APOD data we built. The `rag_text` column is what
we'll embed — it combines the title and explanation for richer context.

In [10]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(sorted(Path('../../storage/gold/apod').glob('apod_*.parquet'))[-1])
print(f"Loaded {len(df)} APOD entries")
print(f"\nSample rag_text:")
print(df['rag_text'].iloc[10][:300])

Loaded 1289 APOD entries

Sample rag_text:
HD 163296: Jet from a Star in Formation. How are jets created during star formation? No one is sure, although recent images of the young star system HD 163296 are quite illuminating. The central star in the featured image is still forming but seen already surrounded by a rotating disk and an outward


## 5. Embed a small sample first

Before embedding all 1,289 entries, test on 5 to make sure
everything works correctly.

In [11]:
sample = df.head(5)
texts = sample['rag_text'].tolist()

print("Embedding 5 sample entries...")
vectors = model.encode(texts, show_progress_bar=True)

print(f"\nVectors shape: {vectors.shape}")
print(f"  {len(texts)} texts × {vectors.shape[1]} dimensions each")

Embedding 5 sample entries...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Vectors shape: (5, 384)
  5 texts × 384 dimensions each


## 6. Store in Chroma

Chroma is a vector database. It stores:
- the vector (for searching by similarity)
- the original text (to return as context to Claude)
- metadata (date, title, url — for display)

`PersistentClient` means it saves to disk so you don't
have to re-embed every time you restart.

In [12]:
import chromadb

# connect to Chroma — saves to disk
client = chromadb.PersistentClient(path="../../storage/chroma")

# create a test collection separate from production
test_collection = client.get_or_create_collection("apod_test")

# add the 5 sample entries
metadatas = sample[["title", "media_type", "word_count"]].copy()
metadatas["date"] = sample["date"].dt.strftime("%Y-%m-%d")
metadatas["url"] = sample["url"].fillna("").astype(str)
metadatas["word_count"] = metadatas["word_count"].astype(int)

test_collection.add(
    ids=sample["hash_id"].tolist(),
    embeddings=vectors.tolist(),
    documents=texts,
    metadatas=metadatas.to_dict(orient="records")
)

print(f"Collection has {test_collection.count()} entries")

Collection has 5 entries


## 7. Search the collection

Embed a question and find the most similar entries.
The score represents cosine similarity — higher is more relevant.

In [13]:
question = "galaxies and stars"
question_vector = model.encode(question).tolist()

results = test_collection.query(
    query_embeddings=[question_vector],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"Question: '{question}'\n")
for i in range(len(results["ids"][0])):
    score = round(1 - results["distances"][0][i], 4)
    meta  = results["metadatas"][0][i]
    print(f"[{score}] {meta['date']} — {meta['title']}")
    print(f"       {results['documents'][0][i][:100]}...")
    print()

Question: 'galaxies and stars'

[-0.1389] 2021-06-16 — Scorpius Enhanced
       Scorpius Enhanced. If Scorpius looked this good to the unaided eye, humans might remember it better....

[-0.3677] 2021-06-13 — A Supercell Thunderstorm Over Texas
       A Supercell Thunderstorm Over Texas. Is that a cloud or an alien spaceship?  It's an unusual and som...

[-0.5649] 2021-06-14 — Ganymede from Juno
       Ganymede from Juno. What does the largest moon in the Solar System look like?  Jupiter's moon Ganyme...



## 8. Connect to Claude — full RAG

Now combine retrieval with generation.
The retrieved APOD entries become context for Claude's answer.
Claude can only answer using what's in the context — grounded in real data.

In [17]:
# delete the test collection and build the real one
chroma_client = chromadb.PersistentClient(path="../../storage/chroma")
chroma_client.delete_collection("apod_test")

# load full dataset
full_collection = chroma_client.get_or_create_collection("apod")

# embed all entries in batches
texts     = df["rag_text"].tolist()
ids       = df["hash_id"].tolist()
metadatas = df[["title", "media_type", "word_count"]].copy()
metadatas["date"]       = df["date"].dt.strftime("%Y-%m-%d")
metadatas["url"]        = df["url"].fillna("").astype(str)
metadatas["word_count"] = metadatas["word_count"].astype(int)
metadata_list = metadatas.to_dict(orient="records")

batch_size = 64
for i in range(0, len(texts), batch_size):
    batch_texts    = texts[i:i+batch_size]
    batch_ids      = ids[i:i+batch_size]
    batch_metadata = metadata_list[i:i+batch_size]
    embeddings     = model.encode(batch_texts, show_progress_bar=False).tolist()

    full_collection.add(
        ids=batch_ids,
        embeddings=embeddings,
        documents=batch_texts,
        metadatas=batch_metadata
    )
    print(f"  embedded {min(i+batch_size, len(texts))}/{len(texts)}")

print(f"\nDone! {full_collection.count()} entries in Chroma")

  embedded 64/1289
  embedded 128/1289
  embedded 192/1289
  embedded 256/1289
  embedded 320/1289
  embedded 384/1289
  embedded 448/1289
  embedded 512/1289
  embedded 576/1289
  embedded 640/1289
  embedded 704/1289
  embedded 768/1289
  embedded 832/1289
  embedded 896/1289
  embedded 960/1289
  embedded 1024/1289
  embedded 1088/1289
  embedded 1152/1289
  embedded 1216/1289
  embedded 1280/1289
  embedded 1289/1289

Done! 1289 entries in Chroma


In [21]:
def rag(question: str, n_results: int = 3) -> dict:
    import anthropic
    import os
    from dotenv import load_dotenv

    load_dotenv('../../.env')
    anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

    # step 1 — embed the question
    question_vector = model.encode(question).tolist()

    # step 2 — retrieve relevant APOD entries
    results = full_collection.query(
        query_embeddings=[question_vector],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    # step 3 — build context from retrieved entries
    context_entries = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        text = results["documents"][0][i]
        context_entries.append(f"Date: {meta['date']}\nTitle: {meta['title']}\n{text}")

    context = "\n\n---\n\n".join(context_entries)

    # step 4 — ask Claude with context
    prompt = f"""You are a NASA astronomy assistant. Answer the question 
using ONLY the APOD entries provided below. 
Cite specific entries by title and date in your answer.
If the entries don't contain enough information, say so.

APOD entries:
{context}

Question: {question}"""

    message = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "question": question,
        "retrieved": [
            {
                "date": results["metadatas"][0][i]["date"],
                "title": results["metadatas"][0][i]["title"],
                "score": round(1 - results["distances"][0][i], 4)
            }
            for i in range(len(results["ids"][0]))
        ],
        "answer": message.content[0].text
    }

In [23]:
result = rag("tell me about galaxies")

print(f"Question: {result['question']}")
print(f"\nRetrieved entries:")
for r in result['retrieved']:
    print(f"  [{r['score']}] {r['date']} — {r['title']}")
print(f"\nAnswer:\n{result['answer']}")

Question: tell me about galaxies

Retrieved entries:
  [0.2706] 2024-08-31 — IFN and the NGC 7771 Group
  [0.2411] 2023-04-12 — NGC 206 and the Star Clouds of Andromeda
  [0.2384] 2024-11-28 — NGC 206 and the Star Clouds of Andromeda

Answer:
# Galaxies from APOD Entries

Based on the provided APOD entries, here is what I can share about galaxies:

---

## The NGC 7771 Group
*("IFN and the NGC 7771 Group," August 31, 2024)*

- Located approximately **200 million light-years away** in the constellation **Pegasus**
- **NGC 7771** is a large, edge-on spiral galaxy spanning about **75,000 light-years** across
- **NGC 7769** is another large spiral galaxy visible face-on nearby
- These galaxies are **actively interacting**, making repeated close passes that will eventually result in **galaxy mergers**
- Their interactions are visible through distorted shapes and **faint streams of stars** caused by gravitational tides

---

## The Andromeda Galaxy (M31)
*("NGC 206 and the Star Clouds of And

In [24]:
questions = [
    "tell me about black holes",
    "interstellar objects visiting our solar system",
    "asteroids close to Earth",
    "the James Webb Space Telescope",
    "supernovas and dying stars",
]

for q in questions:
    result = rag(q)
    print(f"\nQ: {q}")
    for r in result['retrieved']:
        print(f"  [{r['score']}] {r['date']} — {r['title']}")
    print(f"Answer preview: {result['answer'][:150]}...")
    print()


Q: tell me about black holes
  [0.1276] 2024-05-07 — Black Hole Accreting with Jet
  [0.1121] 2022-05-01 — First Horizon-Scale Image of a Black Hole
  [0.0577] 2024-05-10 — Simulation: Two Black Holes Merge
Answer preview: # Black Holes: What We Know from APOD

Here's a summary of key black hole topics covered in the provided APOD entries:

---

## 🔭 What Do Black Holes ...


Q: interstellar objects visiting our solar system
  [0.1686] 2025-08-09 — Interstellar Interloper 3I/ATLAS from Hubble
  [0.1228] 2024-12-22 — The Local Fluff
  [0.1227] 2025-07-07 — Interstellar Comet 3I/ATLAS
Answer preview: # Interstellar Objects Visiting Our Solar System

Based on the provided APOD entries, here is what we know about interstellar objects that have visite...


Q: asteroids close to Earth
  [0.2239] 2023-06-30 — Orbits of Potentially Hazardous Asteroids
  [0.1637] 2025-04-25 — Asteroid Donaldjohanson
  [0.0458] 2023-11-04 — Dinkinesh Moonrise
Answer preview: Based on the APOD entries provided, 

It's working! But the scores are negative and low which means the search isn't finding relevant entries. That's because we only embedded 5 entries — the sample — and none of them are about galaxies.